In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [14]:
import re, time, unicodedata, json, requests
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# ── Pilih backend ────────────────────────────────────────
BACKEND = 'groq'  

# Groq (cloud)
GROQ_API_KEY = 'YOUR_GROQ_API_KEY'
GROQ_MODEL   = 'llama3-8b-8192'  

# ── Context window ───────────────────────────────────────
MAX_CHARS = 3500  # ~75% dari 4096 token Llama 3 8B
                  # Naikkan ke 6000+ untuk Llama 3.1 70B/405B

# Data Paths
DATA_PATH = Path('../../datasets')
PROCESSED_PATH = DATA_PATH / 'processed_data/processed.pkl'
VOCABS_PATH = DATA_PATH / 'processed_data/vocabs.pkl'
SPLITS_PATH = DATA_PATH  / 'split_data/split_indices.json'

# ── Entity labels ────────────────────────────────────────
ENTITY_LABELS = [
    'Name', 'Email Address', 'Years of Experience',
    'Companies worked at', 'Designation', 'Skills',
    'Location', 'College Name', 'Degree', 'Graduation Year', 'UNKNOWN'
]

print('Config loaded OK')
print(f'Backend : {BACKEND} | Labels: {len(ENTITY_LABELS)}')

Config loaded OK
Backend : groq | Labels: 11


In [18]:
import pickle

with open(PROCESSED_PATH, 'rb') as f: 
    processed = pickle.load(f)
with open(VOCABS_PATH, 'rb') as f: 
    vocabs = pickle.load(f)
with open(SPLITS_PATH, 'rb') as f: 
    splits = json.load(f)

idx_train = splits['idx_train']
idx_val= splits['idx_val']
idx_test = splits['idx_test']


ENTITY_LABELS = vocabs['Entity_labels'] + ['UNKNOWN']
id2labels = vocabs['id2label']

print(f'Total records: {len(idx_train)} / {len(idx_val)} / {len(idx_test)}')
print(f'Entity labels: {ENTITY_LABELS}')
p0 = processed[idx_test[0]]
print(f'TEst record {idx_test[0]}:\n{p0["content"][:200]}')
print('5 first span')
for s, e, lbl in p0['spans'][:5]:
    print(f'[{s}:{e}] {lbl:<25} {repr(p0["content"][s:e])}')
    

Total records: 154 / 33 / 33
Entity labels: ['Name', 'Designation', 'Companies worked at', 'Location', 'Email Address', 'College Name', 'Degree', 'Graduation Year', 'Skills', 'Years of Experience', 'UNKNOWN']
TEst record 137:
Dushyant Bhatt
BI / Big Data/ Azure

Hyderabad-Deccan, Telangana, Telangana - Email me on Indeed: indeed.com/r/Dushyant-
Bhatt/140749dace5dc26f

• 10+ years of Experience in Designing, Development, Ad
5 first span
[0:14] Name                      'Dushyant Bhatt'
[37:46] Location                  'Hyderabad'
[98:143] Email Address             'indeed.com/r/Dushyant-\nBhatt/140749dace5dc26f'
[729:738] Companies worked at       'Microsoft'
[1753:1771] Designation               'Software Engineer\n'


In [17]:
def build_prompt(resume_text: str) -> str:
    system_prompt = """You are an expert Named Entity Recognition (NER) system specialized in extracting structured information from resumes and professional profiles.

You MUST follow these rules strictly:
1. Extract ONLY entities explicitly present in the text. Do NOT infer or hallucinate information.
2. Assign exactly ONE label per entity from the allowed label set.
3. If an entity does not clearly belong to any defined label, assign the label UNKNOWN.
4. Return your response as a valid JSON object. No extra explanation, no markdown, no preamble.
5. Each key in the JSON must map to a list of unique string values.
6. Normalize values: strip extra whitespace, proper nouns capitalized.

ALLOWED LABELS:
- Company worked at → Name of a company or organization the person worked at
- Designation → Job title or role
- Skills → Technical skill, tool, language, framework, or soft skill
- Location → City, state, country, or geographic region
- College Name → Name of a university, college, or educational institution
- Degree → Academic degree or qualification
- Graduation Year → Year of graduation (4-digit year)
- Email → Valid email address
- Name → Full name of the candidate
- Years of Experience → Duration of professional experience
- UNKNOWN → Entity present but does not fit any label above

OUTPUT FORMAT (strict JSON):
{
  "NAME": [], "EMAIL": [], "EXPERIENCE": [], "COMPANY": [],
  "DESIGNATION": [], "SKILL": [], "LOCATION": [], "COLLEGE": [],
  "DEGREE": [], "GRAD_YEAR": [], "UNKNOWN": []
}"""

    return (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}\n\n"
        f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        f"Extract all named entities from the following resume text. Return only the JSON object.\n\n"
        f'RESUME TEXT:\n"""\n{resume_text}\n"""\n\n'
        f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )